# Sequence generation with ViralForsight

## Selection of pairs

In [ ]:
import numpy as np

sequences_with_metadata_fasta = 'Path to the fasta file with the sequences lineage information in the header (fex. >genome_id:some_genome_id|lineage:some_lineage)'
training_sequences_fasta = 'Path to the fasta file with the training sequences'
val_dataset_path = 'Path to the npz file containining the validation dataset (it should contain the series accessions under [\'sample_gids\'])'
save_path = 'Path to save the selected sequences'

N = 16 # Number of pairs to be selected

np.random.seed(42)

In [ ]:
import os 
import pickle as pkl

def read_fasta_w_metadata(fasta_filepath):
    metadata = {}
    with open(fasta_filepath, 'r') as f:
        while True:
            header = f.readline().strip()
            sequence = f.readline().strip()
            if not header:
                break
            metadata_pre = header.split('|')
            metadata_seq = {}
            for element in metadata_pre:
                metadata_seq[element.split(':')[0]] = element.split(':')[1]
            metadata_seq['sequence'] = sequence
            metadata[metadata_seq['genome_id']] = metadata_seq
    return metadata

def read_fasta(fasta_file):
    sequence_types = {}
    with open(fasta_file, 'r') as f:
        first = True
        lines = f.readlines()
        for line in lines:
            if line.startswith('>'):
                if not first:
                    sequence_types[label] = sequence
                else:
                    first = False
                label = line.strip()[1:]
                sequence = ''
            else:
                sequence += line.strip()
        sequence_types[label] = sequence
    return sequence_types

sequences = read_fasta_w_metadata(sequences_with_metadata_fasta)

sequences_per_lineage = {}
training_accessions = read_fasta(training_sequences_fasta)

for accession, sequence in training_accessions.items():
    metadata_vals = sequences[accession]
    if metadata_vals['lineage'] not in sequences_per_lineage:
        sequences_per_lineage[metadata_vals['lineage']] = 0
    sequences_per_lineage[metadata_vals['lineage']] += 1

lineage_per_sequence = {}
for accession, metadata in sequences.items():
    lineage_per_sequence[accession] = metadata['lineage']

val_dataset = np.load(val_dataset_path)
val_accessions = np.array(val_dataset['sample_gids'])

selected_N_vals = []
selected_lineages = []
while len(selected_N_vals) < N:
    index = np.random.randint(0, len(val_accessions))
    val_accession = val_accessions[index]
    accession_val = val_accession[-2]
    try:
        lineage = lineage_per_sequence[accession_val]
        seqs_in_lineage = sequences_per_lineage[lineage]
    except:
        continue
    if seqs_in_lineage > 200:
        if accession_val not in selected_N_vals:
            if selected_lineages.count(lineage) < 5:
                selected_N_vals.append(accession_val)
                selected_lineages.append(lineage)

get_seq = lambda x: sequences[x]['sequence']

val_seqs = list(map(get_seq, selected_N_vals))

os.makedirs(save_path, exist_ok=True)

with open(os.path.join(save_path, f'val{N}.txt'), 'w') as f:
    for vs in val_seqs:
        f.write(vs + '\n')

with open(os.path.join(save_path, f'val{N}.fasta'), 'w') as f:
    for va,vs in zip(selected_N_vals, val_seqs):
        f.write(f'>{va}\n{vs}\n')

gts = {}
for series in val_dataset['sample_gids']:
    if series[3] in selected_N_vals:
        if series[3] not in gts:
            gts[series[3]] = []
        gts[series[3]].append(series[4])

pkl.dump(gts, open(os.path.join(save_path, f'gt{N}_all.pkl'), 'wb'))

## Mutation counts

In [ ]:
sequences_with_metadata_fasta = 'Path to the fasta file with the sequences lineage information in the header (fex. >genome_id:some_genome_id|lineage:some_lineage)'
training_sequences_fasta = 'Path to the fasta file with the training sequences'
selected_vals_fasta = 'Path to the fasta file with the selected sequences'
save_path = 'Path to save the mutation probability vectors'

In [ ]:
import os
import numpy as np

def read_fasta_w_metadata(fasta_filepath):
    metadata = {}
    with open(fasta_filepath, 'r') as f:
        while True:
            header = f.readline().strip()
            sequence = f.readline().strip()
            if not header:
                break
            metadata_pre = header.split('|')
            metadata_seq = {}
            for element in metadata_pre:
                metadata_seq[element.split(':')[0]] = element.split(':')[1]
            metadata_seq['sequence'] = sequence
            metadata[metadata_seq['genome_id']] = metadata_seq
    return metadata

def read_fasta(fasta_file):
    sequence_types = {}
    with open(fasta_file, 'r') as f:
        first = True
        lines = f.readlines()
        for line in lines:
            if line.startswith('>'):
                if not first:
                    sequence_types[label] = sequence
                else:
                    first = False
                label = line.strip()[1:]
                sequence = ''
            else:
                sequence += line.strip()
        sequence_types[label] = sequence
    return sequence_types

def get_mutation_counts(sequences, sequence):
    sequences_array = np.array(sequences)
    mutation_counts = []
    # Iterate over columns (positions)
    for i in range(sequences_array.shape[1]):
        aa = sequence[i]
        column = sequences_array[:, i]
        aa_types, aa_counts = np.unique(column, return_counts=True)
        # mutation_score = 1 - (np.max(aa_counts) / np.sum(aa_counts))
        try:
            mutation_score = float(1 - (aa_counts[aa_types == aa] / np.sum(aa_counts)))
        except:
            mutation_score = 1.0
        mutation_counts.append(mutation_score)
    return mutation_counts

sequences = read_fasta_w_metadata(sequences_with_metadata_fasta)
sequences_per_lineage = {}

training_accessions = read_fasta(training_sequences_fasta)

for accession, sequence in training_accessions.items():
    metadata_vals = sequences[accession]
    if metadata_vals['lineage'] not in sequences_per_lineage:
        sequences_per_lineage[metadata_vals['lineage']] = []
    sequences_per_lineage[metadata_vals['lineage']].append(accession)

lineage_per_sequence = {}
for accession, metadata in sequences.items():
    lineage_per_sequence[accession] = metadata['lineage']

os.makedirs(save_path, exist_ok=True)

selected_N_vals = read_fasta(selected_vals_fasta)

for accession, sequence in selected_N_vals.items():
    np.save(os.path.join(save_path, f'{accession}_mutation_counts.npy'), get_mutation_counts(sequences_per_lineage[lineage_per_sequence[accession]], sequence))

## ESM finetuning

In [ ]:
save_folder = "Path to save the finetuned model"
init_data_path = "Path to the training split (.txt with one sequence per line, no header)"

batch_size = 16

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from datasets import Dataset

def get_training_sequences(path):
    seqs = []
    f = open(path, 'r')
    for line in f:
        seqs.append(line.strip())
    return seqs

tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t33_650M_UR50D")
model = AutoModelForMaskedLM.from_pretrained("facebook/esm2_t33_650M_UR50D")
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm_probability=0.15)


train_sequences = get_training_sequences(init_data_path)
train_tokenized = tokenizer(train_sequences)
train_dataset = Dataset.from_dict(train_tokenized)
train_args = TrainingArguments(
    output_dir=save_folder,
    save_strategy = "epoch",
    learning_rate=1e-4,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_steps=1000,
    report_to="none",
)


trainer = Trainer(
    model,
    train_args,
    train_dataset=train_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)
trainer.train()

## Generation

In [ ]:
sequences_path = 'Path to the fasta file with the sequences to mutate'
ft_path = 'Path to the finetuned model'
output_path = ' Path to store the generated sequences'
mutations_path = 'Path to the generated mutation counts'

total_number = 100000
step = 5000

In [ ]:
import os
import re

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForMaskedLM

from dataclasses import dataclass
from typing import Any, Callable, Dict, List, NewType, Optional, Tuple, Union
from collections.abc import Mapping

import numpy as np
import torch

from transformers.data.data_collator import PreTrainedTokenizerBase

import os

## Collator class

@dataclass
class DataCollatorForMaskedGeneration:

    tokenizer: PreTrainedTokenizerBase
    mlm_probability: Any = 0.15
    max_mask: int = 5
    device: Any = 'cuda'
    pad_to_multiple_of: Optional[int] = None
    tf_experimental_compile: bool = False
    return_tensors: str = "pt"


    def __call__(self, features):
        return self.torch_call(features)

    def torch_call(self, examples: List[Union[List[int], Any, Dict[str, Any]]]) -> Dict[str, Any]:
        batch = self.tokenizer.pad(examples, return_tensors="pt", pad_to_multiple_of=self.pad_to_multiple_of)

        special_tokens_mask = batch.pop("special_tokens_mask", None)
        input_ids, len_ = self.torch_mask_tokens(
            batch["input_ids"], special_tokens_mask=special_tokens_mask
        )
        batch["token_ids"] = batch["input_ids"] # no mask
        batch["input_ids"] = input_ids # has mask
        for key, value in batch.items():
            batch[key] = value[:len_].to(self.device)
        return batch

    def torch_mask_tokens(self, inputs: Any, special_tokens_mask: Optional[Any] = None) -> Tuple[Any, Any]:
        """
        Prepare masked tokens inputs/labels for masked language modeling: 80% MASK, 10% random, 10% original.
        """
        while True:
            if type(self.mlm_probability) == float:
                # We sample a few tokens in each sequence for MLM training (with probability `self.mlm_probability`)
                probability_matrix = torch.full(inputs.shape, self.mlm_probability)
            else:
                probability_matrix = self.mlm_probability.repeat(inputs.shape[0], 1)

            if special_tokens_mask is None:
                special_tokens_mask = [
                    self.tokenizer.get_special_tokens_mask(val, already_has_special_tokens=True) for val in inputs.tolist()
                ]
                special_tokens_mask = torch.tensor(special_tokens_mask, dtype=torch.bool)
            else:
                special_tokens_mask = special_tokens_mask.bool()

            probability_matrix.masked_fill_(special_tokens_mask, value=0.0)
            masked_indices = torch.bernoulli(probability_matrix).bool()
            # labels[~masked_indices] = -100  # We only compute loss on masked tokens

            inputs[masked_indices] = self.tokenizer.convert_tokens_to_ids(self.tokenizer.mask_token)

            masked_indices = (masked_indices.sum(dim=1) < self.max_mask) & (masked_indices.sum(dim=1) >= 1)

            if masked_indices.any():
                return inputs[masked_indices], masked_indices.sum().item()

## Functions

def read_fasta(fasta_file):
    sequence_types = {}
    with open(fasta_file, 'r') as f:
        first = True
        lines = f.readlines()
        for line in lines:
            if line.startswith('>'):
                if not first:
                    sequence_types[label] = sequence
                else:
                    first = False
                label = line.strip()[1:]
                sequence = ''
            else:
                sequence += line.strip()
        sequence_types[label] = sequence
    return sequence_types

def get_index(line):
    return int(re.findall(r'[A-Z]([0-9]+)[A-Z]',line)[0])
def get_mut(line):
    return re.findall(r'[A-Z][0-9]+([A-Z])',line)[0]

def numpy_mask_tokens(inputs, probility_mutation, mask_token_id):
    """
    Prepare masked tokens inputs/labels for masked language modeling: 80% MASK, 10% random, 10% original.
    """
    # Numpy doesn't have bernoulli, so we use a binomial with 1 trial
    masked_indices = np.random.binomial(1, probility_mutation, size=probility_mutation.shape).astype(bool)
    masked_lm_positions = np.where(masked_indices == True)[0]
    # The rest of the time (10% of the time) we keep the masked input tokens unchanged
    inputs[masked_lm_positions] = mask_token_id
    return inputs, masked_lm_positions


class GenerateDataset(Dataset):
    def __init__(self, num_samples, rbd_seq, tokenizer):
        super().__init__()
        self.num_samples = num_samples
        self.rbd_seq = rbd_seq
        self.tokenizer = tokenizer

    def __getitem__(self, index):
        return self.tokenizer(self.rbd_seq)

    def __len__(self):
        return self.num_samples

def dump(seqs, name, path):
    file_name = os.path.join(path, "mutation_{}.txt".format(name))
    print('seq num: {}, save path: {}'.format(len(seqs), file_name))
    f = open(file_name, "w")
    for seq in seqs:
        f.write(seq + "\n")
    f.close()

## Generation
device = torch.device("cuda")

sequences = read_fasta(sequences_path)

for accession, sequence in sequences.items():
    data_root = os.path.join(output_path, accession)
    os.makedirs(data_root, exist_ok=True)
    site_freq_path = os.path.join(mutations_path, f'{accession}_mutation_counts.npy')
    mutation_save_path = os.path.join(data_root, f'{accession}_raw_seqs')

    tokenizer = AutoTokenizer.from_pretrained(ft_path)
    model = AutoModelForMaskedLM.from_pretrained(ft_path).to(device).eval()

    max_len = len(sequence)
    max_mask = 5
    topk = 10
    batch_size = 4

    rbd_seq = sequence
    rbd_id = np.array(tokenizer(rbd_seq)['input_ids'])

    save_steps=np.arange(0, total_number+1, step)[1:]
    os.makedirs(mutation_save_path, exist_ok=True)

    probility_mutation = np.load(site_freq_path)
    probility_mutation = np.append(np.append(np.zeros(1), probility_mutation), np.zeros(1))

    collator = DataCollatorForMaskedGeneration(tokenizer, torch.tensor(probility_mutation), max_mask, device=device)

    dataset = GenerateDataset(num_samples=total_number*100, 
                            rbd_seq=rbd_seq, 
                            tokenizer=tokenizer)
    dataloader = DataLoader(dataset, batch_size=batch_size, collate_fn=collator)

    with torch.no_grad():
        output_seq = set()
        output_seq_tp = set()
        process = 0
        print('starting generation...')
        for i, data in enumerate(dataloader):
            if i % 500 == 0:
                print('>>>>> {} loops, current sequence num: {}'.format(i, len(output_seq)))
            token_ids_ = data['token_ids'].cpu()
            data.pop('token_ids')

            out = model(**data)
            indices = torch.topk(out['logits'], topk, dim = -1).indices.cpu()
            bs = indices.shape[0]
            indices = indices.reshape(-1, topk)
            for _ in range(20):
                token_ids = token_ids_.clone()
                index_ran = np.random.randint(0, topk, size=(indices.shape[0]))
                predict_id = indices[range(indices.shape[0]), index_ran]
                predict_id = predict_id.reshape(bs, -1)

                mask_ = (token_ids == tokenizer.mask_token_id)&(predict_id >= 4)&(predict_id <= 23)
                token_ids[mask_] = predict_id[mask_]

                sequences = tokenizer.batch_decode(token_ids, skip_special_tokens=True, clean_up_tokenization_spaces=True)

                for sequence in sequences:
                    sequence = re.sub(r'\s', '', sequence)
                    if sequence not in output_seq:
                        output_seq.add(sequence)
                        output_seq_tp.add(sequence)

                    if process >= len(save_steps):
                        break
                    if len(output_seq) >= save_steps[process]:
                        print('process {}, output_seq: {}, output_seq_tp: {}'.format(process, len(output_seq), len(output_seq_tp)))
                        dump(output_seq_tp, process, mutation_save_path)
                        process+=1
                        output_seq_tp=set()

                if process >= len(save_steps):
                    break
            if process >= len(save_steps):
                break
        print(total_number, len(output_seq))
